In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 1)Data Preprocessing

In [ ]:
# Read data provided by Kaggle
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

print("\nOriginal dataset info:")
print(train_df.info())

print("\nMissing value statistics:")
missing_data = train_df.isnull().sum()
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data[missing_data > 0],
    'Percentage': missing_percent[missing_data > 0]
})
print(missing_df)

print("\nSurvival statistics:")
print(f"Survival rate: {train_df['Survived'].mean()*100:.2f}%")
print(f"Survivors: {train_df['Survived'].sum()}")
print(f"Deceased: {len(train_df) - train_df['Survived'].sum()}")

# Create a copy
df = train_df.copy()

# Fill missing Age values
age_median = df['Age'].median()
df['Age'].fillna(age_median, inplace=True)

# Fill missing Embarked values
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'].fillna(embarked_mode, inplace=True)

# Fill missing Fare values
df['Fare'].fillna(df['Fare'].median(), inplace=True)

# Create new features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                           labels=['Child', 'Teen', 'Adult', 'Middle_Age', 'Senior'])

# Encode categorical variables
label_encoders = {}
categorical_columns = ['Sex', 'Embarked', 'Age_Group']

for col in categorical_columns:
    le = LabelEncoder()
    df[col + '_Encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Select features
feature_columns = ['Pclass', 'Sex_Encoded', 'Age', 'SibSp', 'Parch', 'Fare',
                    'Embarked_Encoded', 'FamilySize', 'IsAlone']

X = df[feature_columns]
y = df['Survived']

print(f"\nFinal feature set: {len(feature_columns)} features")
print(f"Features: {feature_columns}")
print(f"\nPreprocessed data shape: {X.shape}")

# 2)Decision Tree Visualization

In [ ]:
# Train decision tree
dt_classifier = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    criterion='gini'
)

dt_classifier.fit(X, y)

# Training accuracy
train_predictions = dt_classifier.predict(X)
train_accuracy = accuracy_score(y, train_predictions)

print(f"\nDecision tree model parameters:")
print(f"Max depth: {dt_classifier.max_depth}")
print(f"Min samples split: {dt_classifier.min_samples_split}")
print(f"Min samples leaf: {dt_classifier.min_samples_leaf}")

print(f"\nTraining accuracy: {train_accuracy*100:.2f}%")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': dt_classifier.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nFeature importance:")
print(feature_importance.to_string(index=False))

# Decision tree structure
plt.figure(figsize=(20, 12))
plot_tree(dt_classifier,
          feature_names=feature_columns,
          class_names=['Not Survived', 'Survived'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree Structure (Max Depth=5)', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('decision_tree_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# 3)Apply the five-fold cross validation of your fine-tuned decision tree learning model to the Titanic training data to extract average classification accuracy

In [ ]:
# Perform 5-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
dt_cv_scores = cross_val_score(dt_classifier, X, y, cv=kfold, scoring='accuracy')

print(f"\nDecision tree cross-validation results:")
for i, score in enumerate(dt_cv_scores, 1):
    print(f"Fold {i} accuracy: {score*100:.2f}%")

print(f"\n{'-'*40}")
print(f"Mean cross-validation accuracy: {dt_cv_scores.mean()*100:.2f}%")
print(f"Standard deviation: {dt_cv_scores.std()*100:.2f}%")
print(f"95% confidence interval: [{(dt_cv_scores.mean() - 1.96*dt_cv_scores.std())*100:.2f}%, "
      f"{(dt_cv_scores.mean() + 1.96*dt_cv_scores.std())*100:.2f}%]")

# 4)Apply the five-fold cross validation of your fine-tuned random forest learning model to the Titanic training data to extract average classification accuracy

In [ ]:
# Train random forest
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=15,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X, y)

# Training accuracy
rf_train_predictions = rf_classifier.predict(X)
rf_train_accuracy = accuracy_score(y, rf_train_predictions)

print(f"\nRandom forest model parameters:")
print(f"Number of trees: {rf_classifier.n_estimators}")
print(f"Max depth: {rf_classifier.max_depth}")
print(f"Min samples split: {rf_classifier.min_samples_split}")
print(f"Min samples leaf: {rf_classifier.min_samples_leaf}")

print(f"\nTraining performance:")
print(f"Training accuracy: {rf_train_accuracy*100:.2f}%")

# Random forest feature importance
rf_feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_classifier.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nRandom forest feature importance:")
print(rf_feature_importance.to_string(index=False))

# Perform 5-fold cross-validation
rf_cv_scores = cross_val_score(rf_classifier, X, y, cv=kfold, scoring='accuracy')

print(f"\nRandom forest cross-validation results:")
for i, score in enumerate(rf_cv_scores, 1):
    print(f"Fold {i} accuracy: {score*100:.2f}%")

print(f"\n{'-'*40}")
print(f"Mean cross-validation accuracy: {rf_cv_scores.mean()*100:.2f}%")
print(f"Standard deviation: {rf_cv_scores.std()*100:.2f}%")
print(f"95% confidence interval: [{(rf_cv_scores.mean() - 1.96*rf_cv_scores.std())*100:.2f}%, "
      f"{(rf_cv_scores.mean() + 1.96*rf_cv_scores.std())*100:.2f}%]")

# 5)Which algorithm is better, Decision Tree or Random Forest? What are your observations and conclusions from the algorithm comparison and analysis?


| Metric | Decision Tree | Random Forest |
|--------|--------------|---------------|
| Mean CV Accuracy | 82.27% | 82.27% |
| Standard Deviation | 2.98% | 2.15% ⭐ |
| Min Score | 79.21% | 79.89% |
| Max Score | 87.64% | 85.96% |
| Score Range | 8.43% | 6.07% ⭐ |
| 95% CI | [76.44%, 88.10%] | [78.05%, 86.49%] ⭐ |

While both Decision Tree and Random Forest achieved identical mean cross-validation accuracy of 82.27%, representing excellent performance on the Titanic dataset, Random Forest emerges as the superior algorithm due to its significantly better stability (28% lower standard deviation), higher prediction reliability (narrower confidence interval), and robust ensemble learning characteristics. For the Titanic survival prediction task, Random Forest is recommended for deployment, as its consistent performance and reduced variance provide greater value in practical applications than marginal differences in peak accuracy.